In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt



def unpack_4bit_complex(u8: np.ndarray) -> np.ndarray:
    real = (u8 >> 4).astype(np.int8)
    imag = (u8 & 0x0F).astype(np.int8)
    real[real >= 8] -= 16
    imag[imag >= 8] -= 16
    return real.astype(np.float32) + 1j * imag.astype(np.float32)


#path = "/hdd/32_test/caren2901/1142/260129T143648Z_CHARTS_hdf5/baseband_virtual.h5"
#path = "/hdd/32_test/test1002_2/260210T190928Z_CHARTS_hdf5/baseband_virtual.h5"
#path = "/hdd/32_test/260305T165757Z_CHARTS_hdf5/baseband_virtual.h5"
#path = "/hdd/32_test/260305T164228Z_CHARTS_hdf5/baseband_virtual.h5"
#path = "/hdd/32_test/260319T154442Z_CHARTS_hdf5/baseband_virtual.h5"
#path  = "/hdd/32_test/260402T144724Z_CHARTS_hdf5/baseband_virtual.h5"
path = "/hdd/32_test/260407T152530Z_CHARTS_hdf5/baseband_virtual.h5" # sin shield
n_time = 7000 #samples
#path = "/hdd/32_test/260407T152530Z_CHARTS_hdf5/baseband_virtual.h5" # con shield

In [ ]:
import re
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

txt_file = "/home/jpcontreras/kotekan/test_charts/config/test.txt"

with open(txt_file, "r") as f:
    log_text = f.read()

dt_sec = 1.0

pattern = re.compile(
    r"RX pkt\s+(\d+)\s+\(\+(\d+)\)\s+\|\s+lost\s+(\d+)\s+\(\+(\d+)\)\s+\|\s+([\d.]+)\s+Mb/s\s+\|\s+Samples\s+(\d+)"
)

rows = []
for i, line in enumerate(log_text.strip().splitlines()):
    m = pattern.search(line)
    if m:
        rows.append({
            "idx": i,
            "time_s": i * dt_sec,
            "rx_total": int(m.group(1)),
            "rx_delta": int(m.group(2)),
            "lost_total": int(m.group(3)),
            "lost_delta": int(m.group(4)),
            "rate_mbps": float(m.group(5)),
            "samples": int(m.group(6)),
        })

df = pd.DataFrame(rows)

# consider only from second 2 

#df = df[df["time_s"] >= 3.0].reset_index(drop=True)

# Denominadores
den_total = (df["rx_total"] + df["lost_total"]).replace(0, np.nan)
den_interval = (df["rx_delta"] + df["lost_delta"]).replace(0, np.nan)

# Métricas de packet loss
df["loss_pct_total"] = 100.0 * df["lost_total"] / den_total
df["loss_pct_interval"] = 100.0 * df["lost_delta"] / den_interval

# Opcional: reemplazar NaN iniciales por 0
df["loss_pct_total"] = df["loss_pct_total"].fillna(0.0).astype(float)
df["loss_pct_interval"] = df["loss_pct_interval"].fillna(0.0).astype(float)

print(df[["time_s", "rate_mbps", "loss_pct_total", "loss_pct_interval"]])

mean_rate = df["rate_mbps"].mean()
# Plot 1: Data rate vs tiempo
plt.figure(figsize=(10, 5))
plt.plot(df["time_s"], df["rate_mbps"])
plt.xlabel("Time [s]")
plt.ylabel("Data rate [Mb/s]")
plt.title("Data rate vs Time")
plt.grid(True)
plt.legend([f"Mean: {mean_rate:.4f} Mb/s"])
plt.tight_layout()
plt.show()

mean_loss_total = df["loss_pct_interval"].mean()
# Plot 3: Packet loss por intervalo vs tiempo
plt.figure(figsize=(10, 5))
plt.plot(df["time_s"], df["loss_pct_interval"])
plt.xlabel("Time [s]")
plt.ylabel("Packet loss [%]")
plt.title("Packet loss vs Time")
plt.legend([f"Mean: {mean_loss_total:.4f} %"])  
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
with h5py.File(path, "r") as f:
    dset = f["baseband"]

    if n_time > 0:
        packed = dset[:, :, 5000:5000+n_time]  # shape: (ant, freq, time)
    else:
        packed = dset[:, :, :]


    # Header info
    print("File attributes:")
    for k in f.attrs.keys():
        print(f"  {k}: {f.attrs[k]}")



print("Packed shape:", packed.shape)
chains = unpack_4bit_complex(packed) # shape: (antenna, freq, time)


n_ant = chains.shape[0]
n_freq = chains.shape[1]

print(f"Chains decoded shape: {chains.shape} (Antennas, Freqs, Time)")


frequencies = np.linspace(300, 501.6, n_freq, endpoint=False)

antennas = np.linspace(16, 23, 1, dtype=int)

full_scale = 91.6   
for antenna in antennas:
    frame = chains[antenna, :, :]   # (freq, time)
    acc_len = frame.shape[1]

    power_sum = np.sum(np.abs(frame) ** 2, axis=1)
    spectrum_like_fpga = 10 * np.log10((power_sum / acc_len) + 1) - full_scale

    plt.figure(figsize=(10, 4))
    plt.plot(frequencies, spectrum_like_fpga, lw=0.8)
    plt.xlabel("Frequency [MHz]")
    plt.ylabel("Power [dBm]")
    plt.title(f"Antenna {antenna} – Spectrum with FPGA-like scaling")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
with h5py.File(path, "r") as f:
    dset = f["baseband"]

    if n_time > 0:
        packed = dset[:, :, 5000:5000+n_time]  # shape: (ant, freq, time)
    else:
        packed = dset[:, :, :]


    # Header info
    print("File attributes:")
    for k in f.attrs.keys():
        print(f"  {k}: {f.attrs[k]}")



print("Packed shape:", packed.shape)
chains = unpack_4bit_complex(packed) # shape: (antenna, freq, time)


n_ant = chains.shape[0]
n_freq = chains.shape[1]

print(f"Chains decoded shape: {chains.shape} (Antennas, Freqs, Time)")




In [ ]:
frequencies = np.linspace(300, 501.6, n_freq, endpoint=False)

antennas = np.linspace(16, 23, 1, dtype=int)

full_scale = 91.6   
for antenna in antennas:
    frame = chains[antenna, :, :]   # (freq, time)
    acc_len = frame.shape[1]

    power_sum = np.sum(np.abs(frame) ** 2, axis=1)
    spectrum_like_fpga = 10 * np.log10((power_sum / acc_len) + 1) - full_scale

    plt.figure(figsize=(10, 4))
    plt.plot(frequencies, spectrum_like_fpga, lw=0.8)
    plt.xlabel("Frequency [MHz]")
    plt.ylabel("Power [dBm]")
    plt.title(f"Antenna {antenna} – Spectrum with FPGA-like scaling")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
frequencies = np.linspace(300, 501.6, n_freq, endpoint=False)

antennas = np.linspace(16, 23, 8, dtype=int)

full_scale = 91.6   
for antenna in antennas:
    frame = chains[antenna, :, :]   # (freq, time)
    acc_len = frame.shape[1]

    power_sum = np.sum(np.abs(frame) ** 2, axis=1)
    spectrum_like_fpga = 10 * np.log10((power_sum / acc_len) + 1) - full_scale

    plt.figure(figsize=(10, 4))
    plt.plot(frequencies, spectrum_like_fpga, lw=0.8)
    plt.xlabel("Frequency [MHz]")
    plt.ylabel("Power [dBm]")
    plt.title(f"Antenna {antenna} – Spectrum with FPGA-like scaling")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
I = np.abs(frame) ** 2

plt.figure(figsize=(10, 6))
plt.imshow(
    I,
    aspect="auto",
    origin="lower",
    extent=[0, frame.shape[1], frequencies[0], frequencies[-1] ],
    cmap="viridis"
)
plt.colorbar(label="Intensity (arb. units)")
plt.xlabel("Time sample index")
plt.ylabel("Frequency [MHz]")
plt.title(f"Antenna {antenna} – Waterfall")
plt.tight_layout()
plt.show()


In [ ]:
# Correlation of Antenna 7 with specific antennas (15, 23)
target_pairs = [(13, 29)]
n_plots = len(target_pairs)

fig, axes = plt.subplots(1, n_plots, figsize=(12, 5))
if n_plots == 1:
    axes = [axes] # Ensure iterable if only one plot

frequencies = np.linspace(300, 501.6, n_freq, endpoint=False)

print("Calculating specific correlations...")

for idx, (ant_i, ant_j) in enumerate(target_pairs):
    # Calculate correlation
    corr_spectrum = np.mean(chains[ant_i] * np.conj(chains[ant_j]), axis=1)
    phase = np.angle(corr_spectrum)
    
    ax = axes[idx]
    ax.scatter(frequencies, phase, s=1)
    ax.set_ylim(-np.pi, np.pi)
    ax.set_title(f"Ant {ant_i} vs Ant {ant_j}")
    ax.set_xlabel("Freq [MHz]")
    if idx == 0:
        ax.set_ylabel("Phase [rad]")

plt.tight_layout()
plt.show()

In [ ]:
# Determinar el tamaño de la grilla (ej. 4 columnas)
n_cols = 5
n_rows = int(np.ceil((n_ant - 8) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes_flat = axes.flatten()  # Aplanar para iterar fácilmente

print("Calculating correlations and plotting...")

i = 16 # Antena fija 

plot_idx = 0
for j in range(8, n_ant):
    # Calculate correlation
    corr_spectrum = np.mean((chains[i] * np.conj(chains[j])), axis=1)
    phase = np.angle(corr_spectrum)
    
    ax = axes_flat[plot_idx]
    ax.scatter(frequencies, phase, s=3)
    ax.set_title(f"Ant {i} vs Ant {j}", fontsize=10)

    # Etiquetado básico
    if plot_idx >= (n_ant - 8) - n_cols:
        ax.set_xlabel("Frequency [MHz]", fontsize=7)
    
    plot_idx += 1

# Ocultar subplots vacíos
for k in range(plot_idx, len(axes_flat)):
    axes_flat[k].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Determinar el tamaño de la grilla (ej. 4 columnas)
n_cols = 5
n_rows = int(np.ceil(n_ant / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes_flat = axes.flatten()  # Aplanar para iterar fácilmente

frequencies = np.linspace(300, 501.6, n_freq, endpoint=False)  # Frequency axis

print("Calculating correlations and plotting...")

i = 13 # Antena fija 

for j in range(n_ant):
    # Calculate correlation: Average( Antenna_0 * conj(Antenna_j) ) over time axis
    # chains[i] shape is (freq, time)
    # Result shape is (freq,)
    corr_spectrum = np.mean((chains[i] * np.conj(chains[j])), axis=1)
    corr_spectrum = corr_spectrum / np.mean(chains[i] * np.conj(chains[12]), axis=1)
    phase = np.angle(corr_spectrum)
    corr_lag = np.abs(np.fft.fftshift(np.fft.fft(np.nan_to_num(corr_spectrum))))

    lags = np.fft.fftshift(np.fft.fftfreq(len(corr_spectrum), d=1/3.33))
    
    ax = axes_flat[j]
    ax.plot(lags, corr_lag, "-.")
    #ax.set_ylim(-2*np.pi, 2*np.pi)  
    ax.set_title(f"Ant {i} vs Ant {j}", fontsize=10)

    # Etiquetado básico
    if j >= n_ant - n_cols:
        ax.set_xlabel("Lags [us]", fontsize=7)

# Ocultar subplots vacíos
for k in range(n_ant, len(axes_flat)):
    axes_flat[k].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ALL THE CORRELATIONS
ants = range(8, 32)   # últimas 24 antenas
pairs = []

for i in ants:
    for j in ants:
        if j > i:
            pairs.append((i, j))
n_cols = 6
n_rows = int(np.ceil(len(pairs) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4*n_rows))
axes_flat = axes.flatten()

print("Calculating correlations and plotting...")

for idx, (i, j) in enumerate(pairs):

    corr_spectrum = np.mean(chains[i] * np.conj(chains[j]), axis=1)
    phase = np.angle(corr_spectrum)

    ax = axes_flat[idx]
    ax.scatter(frequencies, phase, s=3)
    ax.set_title(f"Ant {i} vs Ant {j}", fontsize=9)

    if idx >= len(pairs) - n_cols:
        ax.set_xlabel("Frequency [MHz]", fontsize=7)

# ocultar plots vacíos
for k in range(len(pairs), len(axes_flat)):
    axes_flat[k].axis('off')

plt.tight_layout()
plt.show()
